# Alice selects two prime numbers p and q and send n=p*q to Bob

In [1]:
# Example small primes (for demo). For real security, use large primes.
p_num = 31
q_num = 43

n = p_num * q_num
print(f"Alice sends n = p*q = {n} to Bob.")

Alice sends n = p*q = 1333 to Bob.


In [2]:
print(f"{p_num} mod 4 : {p_num % 4}")
print(f"{q_num} mod 4 : {q_num % 4}")
print(f"3 mod 4 : {3 % 4}")

31 mod 4 : 3
43 mod 4 : 3
3 mod 4 : 3


## Bob selects a message x and send calculate y = x^2 mod n, then send to Alice

In [3]:
# x = 1414213562373095048 #1414213562373095048
random_a = 100
A = (int)(random_a**2) % n
print(f"A: {A}")
# print(f"Bob select a message x={x} and send to Alice y={y}")

A: 669


## Alice computes X_p and X_q using p and q

In [4]:
fsr_p = (int)((p_num+1)/4)
fsr_q = (int)((q_num+1)/4)
x_p = pow(A, fsr_p, p_num)
x_q = pow(A, fsr_q, q_num)

print(f"The value of x_p congruent to +/- {x_p} (mod {p_num}) ")
print(f"The value of x_q congruent to +/- {x_q} (mod {q_num}) ")

The value of x_p congruent to +/- 7 (mod 31) 
The value of x_q congruent to +/- 14 (mod 43) 


## Chinese Remainder Theorem (CRT)

In [5]:
# Python 3.x
from functools import reduce

def chinese_remainder(n, a):
    total = 0
    prod = reduce(lambda x, y: x * y, n)

    for n_i, a_i in zip(n, a):
        p = prod // n_i
        total += a_i * mul_inv(p, n_i) * p

    return total % prod

def mul_inv(a, b):
    """Multiplicative inverse of a modulo b (assuming gcd(a,b)=1)."""
    b0 = b
    x0, x1 = 0, 1
    if b == 1:
        return 1
    while a > 1:
        q = a // b
        a, b = b, a % b
        x0, x1 = x1 - q * x0, x0
    if x1 < 0:
        x1 += b0
    return x1

In [6]:
prime_nums = [p_num, q_num]

roots = []
for sign_p in (1, -1):
    for sign_q in (1, -1):
        a = [sign_p * x_p, sign_q * x_q]
        r = chinese_remainder(prime_nums, a)
        roots.append(r)

print("Four square roots of A modulo n:")
for r in roots:
    print(f"  beta = {r}")

Four square roots of A modulo n:
  beta = 100
  beta = 1061
  beta = 272
  beta = 1233


In [7]:
def gcd(a, b):
    """Greatest Common Divisor (Euclid)."""
    a, b = abs(a), abs(b)
    while b:
        a, b = b, a % b
    return a

# Students to implement the part where Alice picks one answer and send to Bob
# Bob to use gcd to confirm who is the winner
## Really he just tests gcd (α ± β, n) and sees if he gets something other than 1 or n.

In [8]:
alpha = random_a
print(f"alpha = {alpha}, n = {n}")

# Alice chooses one of the 4 roots to send back as beta.
# For the demo, pick a root that reveals a factor (try 272).
beta = 272
print(f"Alice sends beta = {beta}")

alpha = 100, n = 1333
Alice sends beta = 272


In [9]:
print(f"gcd(alpha + beta, n) = {gcd(alpha + beta, n)}")

gcd(alpha + beta, n) = 31


In [10]:
print(f"gcd(alpha - beta, n) = {gcd(alpha - beta, n)}")

gcd(alpha - beta, n) = 43


In [11]:
# --- Clean result summary (good for screenshots) ---
print("\n=== Summary ===")
print(f"p = {p_num}, q = {q_num}, n = {n}")
print(f"Bob chose alpha = {alpha}")
print(f"A = alpha^2 mod n = {A}")
print(f"Roots mod p: ±{x_p} (mod {p_num})")
print(f"Roots mod q: ±{x_q} (mod {q_num})")
print(f"Four roots beta (mod n): {roots}")

g1 = gcd(alpha + beta, n)
g2 = gcd(alpha - beta, n)
print(f"Alice sent beta = {beta}")
print(f"gcd(alpha + beta, n) = {g1}")
print(f"gcd(alpha - beta, n) = {g2}")

factor = None
if 1 < g1 < n:
    factor = g1
elif 1 < g2 < n:
    factor = g2

if factor is None:
    print("Result: Bob learns nothing (gcd is 1 or n). Alice wins this round.")
else:
    print(f"Result: Bob finds a non-trivial factor of n: {factor}")
    print(f"Other factor: {n // factor}")


=== Summary ===
p = 31, q = 43, n = 1333
Bob chose alpha = 100
A = alpha^2 mod n = 669
Roots mod p: ±7 (mod 31)
Roots mod q: ±14 (mod 43)
Four roots beta (mod n): [100, 1061, 272, 1233]
Alice sent beta = 272
gcd(alpha + beta, n) = 31
gcd(alpha - beta, n) = 43
Result: Bob finds a non-trivial factor of n: 31
Other factor: 43
